# 접촉 전부터 모델의 원판 박스가 아래로 치우칠까?

검증 6명·6편의 무릎 통과부터 바·몸 접촉까지 판독한 56장을 사용합니다. 33번 실험에서 원본 영상을 시간순으로 재생해 선택한 640 박스와 수동 외곽을 같은 원본 좌표로 대조합니다. 각 선수의 1차 접촉 프레임을 0으로 둡니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.contact_edge_run import edge_frames, summarize_edges, plot_edge_traces, plot_contact_examples

pairs = pd.read_csv(ROOT / "outputs/plate_resolution/2026-11-24/paired_frames.csv")
frames = edge_frames(pairs)
output = ROOT / "outputs/contact_edge_run" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
cols = ["person_id", "video_id", "frame_index", "offset", "box_source", "source_640",
        "gt_y1", "gt_y2", "y1_640", "y2_640", "bottom", "top", "center", "bottom_height_pct"]
frames[cols].to_csv(output / "frames.csv", index=False)

## 접촉을 포함해 거꾸로 세기

오차는 모델 y에서 수동 y를 뺀 값입니다. 양수면 화면 아래쪽입니다. `bottom_run`은 아랫변이 아래로 치우진 접촉 포함 연속 장수이고 `bottom_run_before`는 접촉을 빼고 직전 장에서 거꾸로 센 길이입니다. `all_run`은 윗변·아랫변·박스 중심이 동시에 양수인 접촉 포함 길이입니다. 누락이나 0에서 끊습니다. 첫 판독 프레임까지 이어지면 그 이전은 알 수 없습니다.

In [ ]:
summary = summarize_edges(frames)
sensitivity = summarize_edges(frames, threshold_px=1)
summary.to_csv(output / "summary.csv", index=False)
sensitivity.to_csv(output / "sensitivity_1px.csv", index=False)
display(summary.round(2))
display(sensitivity[["person_id", "bottom_run", "all_run"]].rename(columns={
    "bottom_run": "아랫변 1 px 초과", "all_run": "세 지점 모두 1 px 초과"}))

## 접촉에 맞춘 세 경계의 시간 변화

원본 px 단위입니다. 작은 변화도 보이도록 선수별 y축 눈금이 다릅니다. 원판 크기 차이를 참고하려면 `bottom_height_pct`에서 수동 박스 높이 대비 아랫변 차이(%)를 확인합니다. 1 px 초과 표는 소수점 차이에 대한 민감도 비교이며 허용 오차 기준은 아닙니다.

In [ ]:
fig = plot_edge_traces(frames)
fig.savefig(output / "edge_traces.png", dpi=150)
display(fig)
plt.close(fig)

## 서로 다른 두 사례의 원본 비교

b02와 c04의 접촉 2장 전부터 접촉까지를 봅니다. 금색 점선은 수동 외곽, 파란 실선은 입력 640에서 선택한 박스입니다. `people`을 바꾸면 다른 선수도 확인할 수 있습니다.

In [ ]:
fig = plot_contact_examples(frames, people=("b02", "c04"))
fig.savefig(output / "contact_examples.png", dpi=140)
display(fig)
plt.close(fig)
print(output)